# AgroLens — Guava Leaf Disease Classifier Training Notebook

Trains a MobileNetV2-based leaf disease classifier on a guava leaf dataset (GLDD), evaluates it,
saves it, and includes a real-time image-upload inference tester.

**Important note before you run this:** guava is not part of the standard PlantVillage
dataset, so this notebook doesn't assume the `color/grayscale/segmented` folder layout
from before. Section 2 below auto-detects your dataset's actual folder structure instead.
If auto-detection fails, print your folder's contents (`os.listdir(DATASET_DIR)`) and tell
me what it looks like so I can adjust the loading logic to match exactly.

**How to use this notebook:**
1. Set `DATASET_DIR` in the Configuration cell to wherever you unzipped the guava leaf dataset (GLDD).
2. Run all cells top to bottom (`Kernel > Restart & Run All`).
3. Use the last section to upload any leaf photo and see a live prediction.

This notebook trains on your **laptop/Colab GPU**, not on the ESP32-CAM. The ESP32-CAM only
runs the final quantized model (a later conversion step, not covered here).


## 0. Install & import dependencies

In [ ]:
# Run this once. Restart the kernel if TensorFlow was just installed.
# !pip install tensorflow scikit-learn matplotlib seaborn pillow ipywidgets --quiet


In [ ]:
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


: 

## 1. Configuration — EDIT THIS CELL

GLDD typically contains classes such as healthy leaves and various disease/condition categories
(e.g. red rust, fruit fly damage, canker, dot, mummification, etc. depending on the exact release).
Section 2 below reads whatever folder names your download actually uses, so no changes are
needed there regardless of naming.

Set `DATASET_DIR` to the folder where you unzipped the guava leaf dataset (GLDD).

`FILTER_KEYWORD` is left as `None` since this is already a guava leaf-only dataset — no need
to filter by crop name like the tomato notebook did. Leave it `None` unless you specifically
want to keep only some of the classes inside this dataset (e.g. only certain disease types).


In [ ]:
# ============================================================
# EDIT THESE VALUES
# ============================================================

DATASET_DIR = r"C:\path\to\your\guava\dataset"   # <-- CHANGE THIS. Use an r"..." raw string
                                                        # on Windows so backslashes aren't misread.

FILTER_KEYWORD = None        # set to a substring (e.g. "leaf") to keep only matching class folders.
                              # Leave as None to use every class folder found.

IMG_SIZE = (96, 96)          # 96x96 keeps the model small enough for later TFLite Micro conversion.
                              # Use 160x160 or 224x224 only if you plan to prune/quantize aggressively.

BATCH_SIZE = 32
EPOCHS = 15
SEED = 42

WORK_DIR = "./agrolens_guava_work"      # scratch folder this notebook creates
FILTERED_DIR = os.path.join(WORK_DIR, "filtered_dataset")
SPLIT_DIR = os.path.join(WORK_DIR, "split_dataset")
MODEL_SAVE_PATH = os.path.join(WORK_DIR, "agrolens_guava_model.keras")
MODEL_SAVEDMODEL_PATH = os.path.join(WORK_DIR, "agrolens_guava_savedmodel")

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.makedirs(WORK_DIR, exist_ok=True)
assert os.path.isdir(DATASET_DIR), f"DATASET_DIR does not exist: {DATASET_DIR}. Fix the path above."
print("Config set. Dataset dir:", DATASET_DIR)


## 2. Auto-detect dataset structure

Checks for two common layouts:

1. **Already split**: `DATASET_DIR/train/<class>/`, `DATASET_DIR/val or valid/<class>/`, `DATASET_DIR/test/<class>/`
2. **Class folders at some nesting depth**: searches recursively downward from `DATASET_DIR`
   until it finds a folder whose direct subfolders themselves contain images — this handles
   both simple layouts (`DATASET_DIR/<class>/*.jpg`) and deeply nested ones like GLDD's
   `DATASET_DIR/guava/guava/leaf/<class>/*.jpg`, without needing to know the exact depth
   in advance.

It prints what it found. If detection fails, run `print(os.listdir(DATASET_DIR))` yourself,
share the output, and the loading logic can be adjusted to match.


In [ ]:
def looks_like_class_folder(path):
    """A folder 'looks like' a class folder if it directly contains image files."""
    if not os.path.isdir(path):
        return False
    exts = (".jpg", ".jpeg", ".png", ".bmp")
    for f in os.listdir(path)[:20]:
        if f.lower().endswith(exts):
            return True
    return False


def find_class_root(root_dir, max_depth=6):
    """
    Recursively search downward from root_dir for the directory whose direct
    subfolders are themselves class folders (i.e. contain images directly).
    Handles datasets nested arbitrarily deep, like GLDD's guava/guava/leaf/<class>/.
    Returns the found directory, or None if nothing matched within max_depth.
    """
    queue = [(root_dir, 0)]
    while queue:
        current, depth = queue.pop(0)
        if depth > max_depth:
            continue
        subdirs = [d for d in os.listdir(current) if os.path.isdir(os.path.join(current, d))]
        if not subdirs:
            continue
        class_like = [d for d in subdirs if looks_like_class_folder(os.path.join(current, d))]
        # Consider it the class root if at least 2 subfolders directly hold images
        if len(class_like) >= 2:
            return current
        for d in subdirs:
            queue.append((os.path.join(current, d), depth + 1))
    return None


top_level = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))]

DATASET_MODE = None
PRESPLIT_DIRS = {}

# Case 1: already split into train/val/test
train_candidates = [d for d in top_level if d.lower() in ("train", "training")]
val_candidates = [d for d in top_level if d.lower() in ("val", "valid", "validation")]
test_candidates = [d for d in top_level if d.lower() in ("test", "testing")]

if train_candidates and (val_candidates or test_candidates):
    DATASET_MODE = "presplit"
    PRESPLIT_DIRS["train"] = os.path.join(DATASET_DIR, train_candidates[0])
    PRESPLIT_DIRS["val"] = os.path.join(DATASET_DIR, val_candidates[0]) if val_candidates else None
    PRESPLIT_DIRS["test"] = os.path.join(DATASET_DIR, test_candidates[0]) if test_candidates else None
    print("Detected an already train/val/test split dataset:")
    print(PRESPLIT_DIRS)

# Case 2: recursive search for the class-folder level (handles any nesting depth,
# e.g. GLDD's DATASET_DIR/guava/guava/leaf/<class>/)
else:
    found_root = find_class_root(DATASET_DIR)
    if found_root is not None:
        DATASET_MODE = "flat"
        SOURCE_DIR = found_root
        found_classes = [d for d in os.listdir(SOURCE_DIR) if looks_like_class_folder(os.path.join(SOURCE_DIR, d))]
        print(f"Found class-folder level at: {SOURCE_DIR}")
        print("Classes found:", sorted(found_classes))
    else:
        print("Could not auto-detect the structure within the search depth.")
        print("Top-level contents of DATASET_DIR:", top_level)
        raise RuntimeError(
            "Auto-detection failed. Print os.listdir(DATASET_DIR) (and its subfolders) and "
            "share the output so the loading logic can be adjusted to match your dataset's actual layout."
        )

print("\nDataset mode:", DATASET_MODE)


## 3. Build the working train/val/test split

If your dataset was already split (Case 1 above), this just filters classes if requested
and copies it into a consistent working folder. If it was a flat or nested layout (Cases 2/3),
this performs a fresh stratified 70/15/15 split.


In [ ]:
def clean_dataset(root_dir):
    from PIL import Image
    removed = 0
    for subdir, _, files in os.walk(root_dir):
        for f in files:
            path = os.path.join(subdir, f)
            try:
                img = Image.open(path)
                img.verify()
            except Exception:
                os.remove(path)
                removed += 1
    return removed


def filter_classes(source_dir, dest_dir, keyword):
    all_classes = sorted([d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))])
    if keyword:
        selected = [c for c in all_classes if keyword.lower() in c.lower()]
    else:
        selected = all_classes
    assert len(selected) > 0, "No classes matched FILTER_KEYWORD. Check spelling or set it to None."

    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)
    for c in selected:
        shutil.copytree(os.path.join(source_dir, c), os.path.join(dest_dir, c))
    return selected


def split_dataset(source_dir, dest_dir, split_ratio=(0.7, 0.15, 0.15), seed=SEED):
    assert abs(sum(split_ratio) - 1.0) < 1e-6
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)

    classes = sorted([d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))])
    rng = random.Random(seed)

    for split_name in ["train", "val", "test"]:
        for c in classes:
            os.makedirs(os.path.join(dest_dir, split_name, c), exist_ok=True)

    for c in classes:
        files = os.listdir(os.path.join(source_dir, c))
        rng.shuffle(files)
        n = len(files)
        n_train = int(n * split_ratio[0])
        n_val = int(n * split_ratio[1])

        for f in files[:n_train]:
            shutil.copy(os.path.join(source_dir, c, f), os.path.join(dest_dir, "train", c, f))
        for f in files[n_train:n_train + n_val]:
            shutil.copy(os.path.join(source_dir, c, f), os.path.join(dest_dir, "val", c, f))
        for f in files[n_train + n_val:]:
            shutil.copy(os.path.join(source_dir, c, f), os.path.join(dest_dir, "test", c, f))

    return classes


if DATASET_MODE == "presplit":
    # Copy (optionally filtered) presplit folders into a consistent SPLIT_DIR/train,val,test layout
    if os.path.exists(SPLIT_DIR):
        shutil.rmtree(SPLIT_DIR)
    os.makedirs(SPLIT_DIR, exist_ok=True)

    for split_name, src in PRESPLIT_DIRS.items():
        if src is None:
            continue
        dst = os.path.join(SPLIT_DIR, split_name)
        classes_here = filter_classes(src, dst, FILTER_KEYWORD)

    # If there was no separate val set, carve one out of train
    if PRESPLIT_DIRS.get("val") is None:
        train_dir_tmp = os.path.join(SPLIT_DIR, "train")
        val_dir_tmp = os.path.join(SPLIT_DIR, "val")
        os.makedirs(val_dir_tmp, exist_ok=True)
        rng = random.Random(SEED)
        for c in os.listdir(train_dir_tmp):
            os.makedirs(os.path.join(val_dir_tmp, c), exist_ok=True)
            files = os.listdir(os.path.join(train_dir_tmp, c))
            rng.shuffle(files)
            n_val = max(1, int(len(files) * 0.15))
            for f in files[:n_val]:
                shutil.move(os.path.join(train_dir_tmp, c, f), os.path.join(val_dir_tmp, c, f))

    class_names_found = sorted(os.listdir(os.path.join(SPLIT_DIR, "train")))

else:
    filtered_classes = filter_classes(SOURCE_DIR, FILTERED_DIR, FILTER_KEYWORD)
    removed_count = clean_dataset(FILTERED_DIR)
    print(f"Removed {removed_count} corrupt/unreadable files.")
    class_names_found = split_dataset(FILTERED_DIR, SPLIT_DIR)

print("\nFinal classes:", class_names_found)
for split_name in ["train", "val", "test"]:
    split_path = os.path.join(SPLIT_DIR, split_name)
    if os.path.isdir(split_path):
        total = sum(len(os.listdir(os.path.join(split_path, c))) for c in os.listdir(split_path))
        print(f"{split_name}: {total} images")


## 4. Load data + augmentation

Augmentation is applied only to the training set — it simulates the rotation, lighting,
and noise variation the ESP32-CAM camera will see in the field. Validation and test sets
stay unaugmented so evaluation numbers are honest.


In [ ]:
train_dir = os.path.join(SPLIT_DIR, "train")
val_dir = os.path.join(SPLIT_DIR, "val")
test_dir = os.path.join(SPLIT_DIR, "test")

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="categorical"
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="categorical"
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="categorical", shuffle=False
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)
print("Class names (index order used by the model):", class_names)

data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.06),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(0.1),
])

normalization_layer = layers.Rescaling(1./255)

def prep_train(ds):
    ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def prep_eval(ds):
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = prep_train(train_ds_raw)
val_ds = prep_eval(val_ds)
test_ds = prep_eval(test_ds)


In [ ]:
# Sanity check: visualize a batch of augmented training images
plt.figure(figsize=(10, 10))
for images, labels in train_ds_raw.take(1):
    aug_images = data_augmentation(images, training=True)
    for i in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(aug_images[i].numpy().astype("uint8"))
        plt.title(class_names[int(tf.argmax(labels[i]))], fontsize=9)
        plt.axis("off")
plt.tight_layout()
plt.show()


## 5. Model architecture — MobileNetV2 transfer learning

`alpha=0.75` is used here (not the very smallest 0.35 variant) to give the model enough
capacity to separate visually similar diseases first — you can shrink this back down later
during the ESP32-CAM quantization stage if the model doesn't fit in memory.


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
    alpha=0.75,
)
base_model.trainable = False   # freeze backbone for initial training

model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


## 6. Training

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)


In [ ]:
# Optional gentle fine-tuning pass: unfreeze only the last ~10 layers at a very low
# learning rate. Skip this cell if validation accuracy is already good enough —
# unfreezing too much / too fast causes training instability (a loss spike).

base_model.trainable = True
fine_tune_at = len(base_model.layers) - 10
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

fine_tune_epochs = 5
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=fine_tune_epochs,
    callbacks=callbacks,
)


In [ ]:
def plot_history(h1, h2=None):
    acc = h1.history["accuracy"]
    val_acc = h1.history["val_accuracy"]
    loss = h1.history["loss"]
    val_loss = h1.history["val_loss"]

    if h2 is not None:
        acc += h2.history["accuracy"]
        val_acc += h2.history["val_accuracy"]
        loss += h2.history["loss"]
        val_loss += h2.history["val_loss"]

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(acc, label="train")
    plt.plot(val_acc, label="val")
    plt.title("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(loss, label="train")
    plt.plot(val_loss, label="val")
    plt.title("Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_history(history, history_fine if 'history_fine' in dir() else None)


## 7. Evaluation on the held-out test set

Reports accuracy, precision, recall, F1-score (per class and macro-averaged), and a
confusion matrix.


In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

y_true = []
y_pred = []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=class_names))

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
print(f"Macro Precision: {precision:.4f}  Macro Recall: {recall:.4f}  Macro F1: {f1:.4f}")


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix — test set")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 8. Save the trained model

Saves both a `.keras` file and a TensorFlow SavedModel folder (needed later for TFLite /
TFLite Micro conversion for ESP32-CAM deployment).


In [ ]:
model.save(MODEL_SAVE_PATH)
model.export(MODEL_SAVEDMODEL_PATH)

with open(os.path.join(WORK_DIR, "class_names.txt"), "w") as f:
    for c in class_names:
        f.write(c + "\n")

print("Saved model to:", MODEL_SAVE_PATH)
print("Saved SavedModel export to:", MODEL_SAVEDMODEL_PATH)
print("Saved class names to:", os.path.join(WORK_DIR, "class_names.txt"))
print("\nNext step (separate stage): INT8 quantization + TFLite Micro conversion for ESP32-CAM.")


## 9. Real-time inference — upload an image and get a live prediction

Run the cell below, click "Upload", pick any guava leaf photo, and the model's prediction
(disease name + confidence) displays immediately below it.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import io

uploader = widgets.FileUpload(accept="image/*", multiple=False, description="Upload leaf photo")
output = widgets.Output()

def predict_image(img: Image.Image):
    img_resized = img.convert("RGB").resize(IMG_SIZE)
    arr = np.array(img_resized).astype("float32") / 255.0
    arr = np.expand_dims(arr, axis=0)
    preds = model.predict(arr, verbose=0)[0]
    top_idx = int(np.argmax(preds))
    return class_names[top_idx], float(preds[top_idx]), preds

def on_upload_change(change):
    with output:
        clear_output()
        if not uploader.value:
            return
        uploaded_file = list(uploader.value.values())[0] if isinstance(uploader.value, dict) else uploader.value[0]
        content = uploaded_file["content"] if isinstance(uploaded_file, dict) else uploaded_file.content
        img = Image.open(io.BytesIO(content))

        predicted_class, confidence, all_preds = predict_image(img)

        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"{predicted_class}\n{confidence*100:.1f}% confidence", fontsize=11)
        plt.show()

        print("Full class probabilities:")
        for cname, p in sorted(zip(class_names, all_preds), key=lambda x: -x[1]):
            print(f"  {cname:35s} {p*100:5.1f}%")

uploader.observe(on_upload_change, names="value")
display(uploader, output)


### Notes

- If auto-detection in Section 2 failed, share the output of `os.listdir(DATASET_DIR)` and
  the loading code can be adjusted precisely.
- Radish disease classification has less established literature than PlantVillage/tomato —
  worth discussing explicitly in your paper if this is the dataset you commit to for the
  final project.
- The next stage (not in this notebook) converts `agrolens_guava_model.keras` into an
  INT8-quantized `.tflite` file sized for the ESP32-CAM.
